# Tratamento e Padronização de Dados com Pandas

Este notebook trata **duas bases** de questionário com o **mesmo código**.

| Base | Arquivo | Origem |
|------|---------|--------|
| Base 1 | `../dados/brutos/base1.xlsx` | Respostas do Moodle (colunas genéricas) |
| Base 2 | `../dados/brutos/base2.xlsx` | Versão aprimorada (colunas descritivas) |

**Objetivo:** padronizar turma, data de nascimento, município, empresa, quantidade de moradores e se já trabalhou.


## 1. Importações e configuração

In [ ]:
from pathlib import Path
from typing import Optional, Tuple, Union
import re
import unicodedata

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 120)

# Raiz do projeto (funciona rodando de notebooks/ ou da raiz)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "dados").exists() and (ROOT.parent / "dados").exists():
    ROOT = ROOT.parent

PASTA_BRUTOS = ROOT / "dados" / "brutos"
PASTA_TRATADOS = ROOT / "dados" / "tratados"
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT.resolve())
print("Pastas prontas:", PASTA_BRUTOS.resolve(), PASTA_TRATADOS.resolve())


## 2. Schema padrão e mapeamento de colunas

As duas bases falam das mesmas perguntas, mas com nomes diferentes.
Mapeamos tudo para o mesmo schema:

In [ ]:
COLUNAS_PADRAO = [
    "turma",
    "data_nascimento",
    "municipio",
    "empresa",
    "qtd_pessoas_moradia",
    "ja_trabalhou",
]

# Base 1 (Moodle): Resposta 1..6
MAPA_BASE1 = {
    "Resposta 1": "turma",
    "Resposta 2": "data_nascimento",
    "Resposta 3": "municipio",
    "Resposta 4": "empresa",
    "Resposta 5": "qtd_pessoas_moradia",
    "Resposta 6": "ja_trabalhou",
}

# Base 2 (versão aprimorada): nomes longos das perguntas
MAPA_BASE2 = {
    "(TURMA) Selecione a sua turma": "turma",
    "Digite sua data de nascimento (ex.: 19/06/1999)": "data_nascimento",
    "Digite o município (cidade) em que nasceu (ex.: São Paulo)": "municipio",
    (
        "Digite o nome breve da empresa em que trabalha "
        "(ao invés de inserir São Paulo Tech School, você digitaria SPTECH, por exemplo). "
        "Caso não esteja trabalhando, digite 'Open to Work'."
    ): "empresa",
    "Mora com quantas pessoas (incluindo você)?": "qtd_pessoas_moradia",
    "Já trabalhou antes de entrar na SPTECH?": "ja_trabalhou",
}


def detectar_mapa_colunas(df: pd.DataFrame) -> dict:
    """Escolhe o mapeamento certo conforme as colunas da planilha."""
    cols = set(df.columns)
    if set(MAPA_BASE1).issubset(cols):
        return MAPA_BASE1
    if set(MAPA_BASE2).issubset(cols):
        return MAPA_BASE2
    raise ValueError(
        "Formato de base não reconhecido. Colunas encontradas:\n"
        + "\n".join(map(str, df.columns))
    )


def extrair_respostas(df: pd.DataFrame) -> pd.DataFrame:
    """Renomeia e seleciona apenas as colunas do schema padrão."""
    mapa = detectar_mapa_colunas(df)
    out = df.rename(columns=mapa)[COLUNAS_PADRAO].copy()
    return out.reset_index(drop=True)


print("Schema padrão:", COLUNAS_PADRAO)


## 3. Funções auxiliares de limpeza

In [ ]:
def limpar_texto(valor) -> Optional[str]:
    """Converte para string, remove espaços extras e HTML entities simples."""
    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    texto = texto.replace("&#039;", "'").replace("&amp;", "&").replace("&nbsp;", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto if texto else None


def remover_acentos(texto: str) -> str:
    normalizado = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in normalizado if not unicodedata.combining(c))


def chave_comparacao(texto: str) -> str:
    """Texto em minúsculas, sem acento e sem pontuação — útil para matching."""
    texto = remover_acentos(texto).lower()
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


### 3.1 Turma

In [ ]:
def padronizar_turma(valor) -> Optional[str]:
    texto = limpar_texto(valor)
    if texto is None:
        return None

    chave = chave_comparacao(texto).replace(" ", "")

    # Já no formato esperado
    if chave in {"5adsa", "5adsb"}:
        return chave.upper()

    # Variações sujas da Base 1 (espaço, hífen, ordem invertida, etc.)
    if chave in {
        "adsb", "adsb5", "5adsb", "5ads", "5adb", "5asdb",
        "5adsb", "cincoadsb", "5absb", "be", "b",
    } or "adsb" in chave or chave.endswith("adsb") or chave == "cincoadsb":
        return "5ADSB"

    if "adsa" in chave:
        return "5ADSA"

    # Fallback: se contém 'ads' e 'b', assume 5ADSB
    if "ads" in chave and "b" in chave:
        return "5ADSB"

    return None


### 3.2 Data de nascimento

Aceita formatos comuns e alguns textos em português. Datas incompletas ou inválidas ficam como `NaT`.

In [ ]:
MESES = {
    "janeiro": 1, "fevereiro": 2, "marco": 3, "abril": 4,
    "maio": 5, "junho": 6, "julho": 7, "agosto": 8,
    "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12,
}

NUMEROS_EXTENSO = {
    "um": 1, "uma": 1, "dois": 2, "duas": 2, "tres": 3,
    "quatro": 4, "cinco": 5, "seis": 6, "sete": 7,
    "oito": 8, "nove": 9, "dez": 10, "onze": 11, "doze": 12,
    "treze": 13, "quatorze": 14, "catorze": 14, "quinze": 15,
    "dezesseis": 16, "dezessete": 17, "dezoito": 18, "dezenove": 19,
    "vinte": 20, "trinta": 30,
    "quarenta": 40, "cinquenta": 50, "sessenta": 60,
    "setenta": 70, "oitenta": 80, "noventa": 90,
}


def _ano_extenso(texto: str) -> Optional[int]:
    """Converte expressões como 'dois mil e quatro' ou 'mil novecentos e noventa e nove'."""
    t = chave_comparacao(texto)

    # Ex.: dois mil e 3 / dois mil e quatro / Dois mil e quatro
    m = re.search(r"dois mil(?: e )?(\d+|quatro|cinco|seis|sete|oito|nove|tres|um|dois)?", t)
    if m:
        resto = m.group(1)
        if resto is None:
            return 2000
        if resto.isdigit():
            return 2000 + int(resto)
        return 2000 + NUMEROS_EXTENSO.get(resto, 0)

    # Ex.: mil novecentos e noventa e nove
    if "mil novecentos" in t:
        # pega os últimos números por extenso após 'novecentos'
        partes = t.split("novecentos")[-1]
        total = 1900
        for token in partes.split():
            if token == "e":
                continue
            if token.isdigit():
                total += int(token)
            elif token in NUMEROS_EXTENSO:
                total += NUMEROS_EXTENSO[token]
        return total if total > 1900 else None

    return None


def _parse_data_textual(texto: str) -> Optional[pd.Timestamp]:
    t = chave_comparacao(texto)

    # "25 de junho de 2004"
    m = re.search(r"(\d{1,2})\s+de\s+([a-z]+)\s+de\s+(\d{4})", t)
    if m:
        dia, mes_nome, ano = int(m.group(1)), m.group(2), int(m.group(3))
        mes = MESES.get(mes_nome)
        if mes:
            return pd.Timestamp(year=ano, month=mes, day=dia)

    # "trinta de dezembro de mil novecentos e noventa e nove"
    m = re.search(r"([a-z]+)\s+de\s+([a-z]+)\s+de\s+(.+)", t)
    if m and m.group(1) in NUMEROS_EXTENSO and m.group(2) in MESES:
        dia = NUMEROS_EXTENSO[m.group(1)]
        mes = MESES[m.group(2)]
        ano = _ano_extenso(m.group(3))
        if ano:
            return pd.Timestamp(year=ano, month=mes, day=dia)

    # Apenas ano por extenso / número
    ano = _ano_extenso(t)
    if ano:
        # sem dia/mês confiável → inválido para data completa
        return pd.NaT

    return pd.NaT


def padronizar_data(valor) -> Optional[pd.Timestamp]:
    texto = limpar_texto(valor)
    if texto is None:
        return pd.NaT

    # Só o ano (ex.: 2004, 1999) — incompleto
    if re.fullmatch(r"\d{4}", texto):
        return pd.NaT

    # ddmmyyyy sem separador (ex.: 15052002)
    if re.fullmatch(r"\d{8}", texto):
        try:
            return pd.to_datetime(texto, format="%d%m%Y", errors="raise")
        except Exception:
            return pd.NaT

    # Troca hífen por barra para unificar
    candidato = texto.replace("-", "/")

    formatos = ["%d/%m/%Y", "%Y/%m/%d", "%d/%m/%y"]
    for fmt in formatos:
        try:
            dt = pd.to_datetime(candidato, format=fmt, errors="raise")
            # rejeita datas absurdas (antes de 1950 ou no futuro distante)
            if dt.year < 1950 or dt.year > 2015:
                return pd.NaT
            return dt
        except Exception:
            continue

    # Tentativa textual
    dt = _parse_data_textual(texto)
    if pd.notna(dt):
        if dt.year < 1950 or dt.year > 2015:
            return pd.NaT
        return dt

    return pd.NaT


### 3.3 Município

In [ ]:
# Valores que claramente não são município
MUNICIPIOS_INVALIDOS = {
    "maternidade", "brasil", "gotham city", "gotham",
}

# Alias → nome oficial
ALIAS_MUNICIPIO = {
    "sao paulo": "São Paulo",
    "sp": "São Paulo",
    "sao paulo sp": "São Paulo",
    "sao paulo sao paulo": "São Paulo",
    "sp sp": "São Paulo",
    "sao bernardo do campo": "São Bernardo do Campo",
    "santo andre": "Santo André",
    "guarulhos": "Guarulhos",
    "osasco": "Osasco",
    "barueri": "Barueri",
    "embu das artes": "Embu das Artes",
    "carapicuiba": "Carapicuíba",
    "paulista": "Paulista",
    "paulista pe": "Paulista",
}


def padronizar_municipio(valor) -> Optional[str]:
    texto = limpar_texto(valor)
    if texto is None:
        return None

    # Remove UF / sufixos comuns: "São Paulo - SP", "São Paulo, São Paulo"
    texto = re.split(r"\s*[-,]\s*", texto)[0].strip()
    chave = chave_comparacao(texto)

    if chave in MUNICIPIOS_INVALIDOS:
        return None

    if chave in ALIAS_MUNICIPIO:
        return ALIAS_MUNICIPIO[chave]

    # Title case como fallback
    return texto.title()


### 3.4 Empresa

In [ ]:
EMPRESAS_CANONICAS = [
    ("stefanini", "Stefanini"),
    ("evertec", "Evertec"),
    ("sinqia", "Evertec"),  # antiga Sinqia
    ("ecorodovias", "Ecorodovias"),
    ("dotz", "Dotz"),
    ("targit", "Targit"),
    ("finaya", "Finaya"),
    ("dock", "Dock"),
    ("safra", "Safra"),
    ("simpress", "Simpress"),
    ("btg", "BTG Pactual"),
    ("semantix", "Semantix"),
    ("avanade", "Avanade"),
    ("avanad", "Avanade"),  # typo
    ("deloitte", "Deloitte"),
    ("c6", "C6 Bank"),
    ("motiva", "Motiva"),
    ("ccr", "Motiva"),  # Motiva antiga CCR
]

SEM_EMPRESA = {
    "nenhuma", "open to work", "opentowork", "nao trabalho",
    "desempregado", "sem empresa",
}


def padronizar_empresa(valor) -> Optional[str]:
    texto = limpar_texto(valor)
    if texto is None:
        return None

    # Remove aspas soltas
    texto = texto.strip("'\"")
    chave = chave_comparacao(texto)

    if chave in SEM_EMPRESA or "open to work" in chave:
        return "Open to Work"

    for trecho, canonico in EMPRESAS_CANONICAS:
        if trecho in chave:
            return canonico

    # Fallback: primeira palavra significativa em Title Case
    return texto.split(",")[0].strip().title()


### 3.5 Quantidade de pessoas na moradia

In [ ]:
def padronizar_qtd_pessoas(valor) -> Optional[int]:
    texto = limpar_texto(valor)
    if texto is None:
        return None

    chave = chave_comparacao(texto)

    # "Só uma", "Uma"
    if chave in {"so uma", "uma", "somente uma", "apenas uma"}:
        return 1

    # Extrai o primeiro número da string (ex.: "3 (incluindo eu)", "atualmente 4")
    m = re.search(r"\d+", texto)
    if m:
        n = int(m.group())
        return n if 1 <= n <= 20 else None

    # Número por extenso
    for palavra, n in NUMEROS_EXTENSO.items():
        if palavra in chave.split():
            return n

    return None


### 3.6 Já trabalhou (Sim/Não)

In [ ]:
def padronizar_ja_trabalhou(valor) -> Optional[str]:
    texto = limpar_texto(valor)
    if texto is None:
        return None

    chave = chave_comparacao(texto)

    # Afirmativos (inclui inglês, espanhol, emoji e frases)
    if chave in {"sim", "si", "yes", "of course", "👍"} or texto.strip() == "👍":
        return "Sim"
    if any(p in chave for p in ["sim", "trabalhei", "of course", "yes"]):
        # "Sim, trabalhei sim" / "Sim."
        if not any(n in chave for n in ["nao", "nunca", "nenhuma"]):
            return "Sim"

    # Negativos (inclui japonês いいえ = não)
    if chave in {"nao", "no", "いいえ", "nenhuma vez"} or "nao" in chave or "nunca" in chave:
        return "Não"
    if "いいえ" in str(valor):
        return "Não"

    # Emoji positivo restante
    if "👍" in str(valor):
        return "Sim"

    return None


## 4. Pipeline único (serve para as 2 bases)

In [ ]:
def tratar_base(df_bruto: pd.DataFrame) -> pd.DataFrame:
    """Aplica o mesmo tratamento em qualquer uma das duas bases."""
    df = extrair_respostas(df_bruto)

    tratado = pd.DataFrame({
        "turma": df["turma"].map(padronizar_turma),
        "data_nascimento": df["data_nascimento"].map(padronizar_data),
        "municipio": df["municipio"].map(padronizar_municipio),
        "empresa": df["empresa"].map(padronizar_empresa),
        "qtd_pessoas_moradia": df["qtd_pessoas_moradia"].map(padronizar_qtd_pessoas),
        "ja_trabalhou": df["ja_trabalhou"].map(padronizar_ja_trabalhou),
    })

    # Formata data como texto DD/MM/AAAA para exportação limpa
    tratado["data_nascimento"] = tratado["data_nascimento"].dt.strftime("%d/%m/%Y")

    return tratado


def carregar_e_tratar(caminho: Union[Path, str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Lê o Excel e devolve (bruto_padronizado_colunas, tratado)."""
    bruto = pd.read_excel(caminho)
    respostas = extrair_respostas(bruto)
    tratado = tratar_base(bruto)
    return respostas, tratado


print("Pipeline definido: tratar_base() / carregar_e_tratar()")


## 5. Processar Base 1 e Base 2

In [ ]:
bruto1, base1 = carregar_e_tratar(PASTA_BRUTOS / "base1.xlsx")
bruto2, base2 = carregar_e_tratar(PASTA_BRUTOS / "base2.xlsx")

print("Base 1 — antes:", bruto1.shape, "| depois:", base1.shape)
print("Base 2 — antes:", bruto2.shape, "| depois:", base2.shape)


### 5.1 Comparativo Base 1 (antes × depois)

In [ ]:
print("=== BASE 1 — ANTES (amostra) ===")
display(bruto1.head(10))

print("=== BASE 1 — DEPOIS (amostra) ===")
display(base1.head(10))

print("Nulos após tratamento (Base 1):")
print(base1.isna().sum())


### 5.2 Comparativo Base 2 (antes × depois)

In [ ]:
print("=== BASE 2 — ANTES (amostra) ===")
display(bruto2.head(10))

print("=== BASE 2 — DEPOIS (amostra) ===")
display(base2.head(10))

print("Nulos após tratamento (Base 2):")
print(base2.isna().sum())


### 5.3 Checagem de valores padronizados

In [ ]:
def resumo_coluna(df: pd.DataFrame, col: str, nome: str):
    print(f"\n[{nome}] {col} — valores únicos:")
    print(df[col].value_counts(dropna=False).head(15))


for nome, df in [("Base 1", base1), ("Base 2", base2)]:
    print("\n" + "=" * 60)
    print(nome)
    for col in COLUNAS_PADRAO:
        resumo_coluna(df, col, nome)


## 6. Exportar bases tratadas

In [ ]:
caminho_base1 = PASTA_TRATADOS / "Base1_tratada.xlsx"
caminho_base2 = PASTA_TRATADOS / "Base2_tratada.xlsx"

base1.to_excel(caminho_base1, index=False)
base2.to_excel(caminho_base2, index=False)

print("Arquivos gerados:")
print(" -", caminho_base1.resolve())
print(" -", caminho_base2.resolve())

display(base1)
display(base2)


## Resumo do que o código faz

1. **Detecta** automaticamente se a planilha é Base 1 ou Base 2 pelo nome das colunas.
2. **Renomeia** tudo para o mesmo schema: `turma`, `data_nascimento`, `municipio`, `empresa`, `qtd_pessoas_moradia`, `ja_trabalhou`.
3. **Padroniza** cada campo (turma, datas em vários formatos, cidades, empresas, números e Sim/Não).
4. **Exporta** `dados/tratados/Base1_tratada.xlsx` e `dados/tratados/Base2_tratada.xlsx`.

Valores impossíveis de interpretar com segurança (ex.: só o ano da data, "Maternidade", "Gotham City") ficam como nulo (`NaN`).
